In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import yaml

from crispdm.data import download_utils_data

In [2]:
download_utils_data.download_microsoft_dataset()

In [3]:
def find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() or (parent / ".git").exists():
            return parent
    raise RuntimeError("Project root not found (pyproject.toml/.git)")


PROJECT_ROOT = find_project_root()
CONFIG_ROOT = PROJECT_ROOT / "config"
CONFIG_DATA = PROJECT_ROOT / "data"
dataset_cfg_path = CONFIG_ROOT / "datasets" / "dataset_config.yml"

with open(dataset_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

ds = cfg["datasets"]["microsoft_security_incident"]

train_rel = ds["paths"]["train"]
test_rel = ds["paths"]["test"]

csv_params = ds.get("csv_params", {})

train_path = PROJECT_ROOT / train_rel
test_path = PROJECT_ROOT / test_rel

print("Train path:", train_path)
print("Test path:", test_path)


# 'data/raw/train/GUIDE_Train.csv'

Train path: K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\data\raw\train\GUIDE_Train.csv
Test path: K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\data\raw\test\GUIDE_Test.csv


In [4]:
df_head = pd.read_csv(
    train_path,
    sep=csv_params.get("sep", ","),
    encoding=csv_params.get("encoding", "utf-8"),
    decimal=csv_params.get("decimal", "."),
    low_memory=csv_params.get("low_memory", False),
    nrows=3,
)
df_head

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,180388628218,0,612,123247,2024-06-04T06:05:15.000Z,7,6,InitialAccess,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,31,6,3
1,455266534868,88,326,210035,2024-06-14T03:01:25.000Z,58,43,Exfiltration,NaN,FalsePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
2,1056561957389,809,58352,712507,2024-06-13T04:52:55.000Z,423,298,InitialAccess,T1189,FalsePositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630


In [5]:
df_head.columns.tolist()
df_head.Category

0    InitialAccess
1     Exfiltration
2    InitialAccess
Name: Category, dtype: object

In [6]:
df_head = pd.read_csv(
    test_path,
    sep=csv_params.get("sep", ","),
    encoding=csv_params.get("encoding", "utf-8"),
    decimal=csv_params.get("decimal", "."),
    low_memory=csv_params.get("low_memory", False),
    nrows=3,
)
df_head

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City,Usage
0,1245540519230,657,11767,87199,2024-06-04T22:56:27.000Z,524,563,LateralMovement,T1021;T1047;T1105;T1569.002,BenignPositive,...,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630,Private
1,1400159342154,3,91158,632273,2024-06-03T12:58:26.000Z,2,2,CommandAndControl,NaN,BenignPositive,...,NaN,0,0,NaN,Suspicious,Suspicious,242,1445,10630,Public
2,1279900255923,145,32247,131719,2024-06-08T03:20:49.000Z,2932,10807,LateralMovement,T1021;T1027.002;T1027.005;T1105,BenignPositive,...,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630,Public


In [7]:
df_head.columns.tolist()
df_head.Category

0      LateralMovement
1    CommandAndControl
2      LateralMovement
Name: Category, dtype: object

In [8]:
ORG_ID = 11
INCIDENT_ID = 417400

query = f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT Id) AS n_distinct_id,
  COUNT(DISTINCT AlertId) AS n_distinct_alert,
  COUNT(DISTINCT CAST(Timestamp AS VARCHAR)) AS n_distinct_ts
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE OrgId = {ORG_ID} AND IncidentId = {INCIDENT_ID}
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_distinct_id,n_distinct_alert,n_distinct_ts
0,4,1,1,1


In [9]:
query = f"""
SELECT DISTINCT
  *
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE OrgId = {ORG_ID} AND IncidentId = {INCIDENT_ID}
ORDER BY Timestamp
"""
incident_df = duckdb.query(query).to_df()

pd.set_option("display.max_columns", None)

incident_df
# incident_df.iloc[0].to_frame("value")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,ActionGrouped,ActionGranular,EntityType,EvidenceRole,DeviceId,Sha256,IpAddress,Url,AccountSid,AccountUpn,AccountObjectId,AccountName,DeviceName,NetworkMessageId,EmailClusterId,RegistryKey,RegistryValueName,RegistryValueData,ApplicationId,ApplicationName,OAuthApplicationId,ThreatFamily,FileName,FolderPath,ResourceIdName,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,CloudLogonSession,Related,98799,138268,360606,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,242,1445,10630
1,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,Ip,Related,98799,138268,30410,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,8,6,3
2,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,User,Impacted,98799,138268,360606,160396,28670,48804,29043,32324,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,242,1445,10630
3,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,CloudLogonRequest,Related,98799,138268,360606,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,242,1445,10630


In [10]:
delim = csv_params.get("sep", ",")

base = f"read_csv_auto('{train_path}', delim='{delim}')"

base_test = f"read_csv_auto('{test_path}', delim='{delim}')"

In [11]:
schema_df = duckdb.query(f"DESCRIBE SELECT * FROM {base}").to_df()
# schema_df, len(schema_df)
schema_df[["column_name", "column_type"]]

,column_name,column_type
0,Id,BIGINT
1,OrgId,BIGINT
2,IncidentId,BIGINT
3,AlertId,BIGINT
4,Timestamp,TIMESTAMP WITH TIME ZONE
5,DetectorId,BIGINT
6,AlertTitle,BIGINT
7,Category,VARCHAR
8,MitreTechniques,VARCHAR
9,IncidentGrade,VARCHAR


In [12]:
schema_df = duckdb.query(f"DESCRIBE SELECT * FROM {base_test}").to_df()
# schema_df, len(schema_df)
schema_df[["column_name", "column_type"]]

,column_name,column_type
0,Id,BIGINT
1,OrgId,BIGINT
2,IncidentId,BIGINT
3,AlertId,BIGINT
4,Timestamp,TIMESTAMP WITH TIME ZONE
5,DetectorId,BIGINT
6,AlertTitle,BIGINT
7,Category,VARCHAR
8,MitreTechniques,VARCHAR
9,IncidentGrade,VARCHAR


In [13]:
schema_df = duckdb.query(f"DESCRIBE SELECT * FROM {base_test}").to_df()
schema_df, len(schema_df)

(           column_name               column_type null   key default extra
 0                   Id                    BIGINT  YES  None    None  None
 1                OrgId                    BIGINT  YES  None    None  None
 2           IncidentId                    BIGINT  YES  None    None  None
 3              AlertId                    BIGINT  YES  None    None  None
 4            Timestamp  TIMESTAMP WITH TIME ZONE  YES  None    None  None
 5           DetectorId                    BIGINT  YES  None    None  None
 6           AlertTitle                    BIGINT  YES  None    None  None
 7             Category                   VARCHAR  YES  None    None  None
 8      MitreTechniques                   VARCHAR  YES  None    None  None
 9        IncidentGrade                   VARCHAR  YES  None    None  None
 10       ActionGrouped                   VARCHAR  YES  None    None  None
 11      ActionGranular                   VARCHAR  YES  None    None  None
 12          EntityType  

In [14]:
schema_df = duckdb.query(f"DESCRIBE SELECT * FROM {base}").to_df()
schema_df, len(schema_df)

(           column_name               column_type null   key default extra
 0                   Id                    BIGINT  YES  None    None  None
 1                OrgId                    BIGINT  YES  None    None  None
 2           IncidentId                    BIGINT  YES  None    None  None
 3              AlertId                    BIGINT  YES  None    None  None
 4            Timestamp  TIMESTAMP WITH TIME ZONE  YES  None    None  None
 5           DetectorId                    BIGINT  YES  None    None  None
 6           AlertTitle                    BIGINT  YES  None    None  None
 7             Category                   VARCHAR  YES  None    None  None
 8      MitreTechniques                   VARCHAR  YES  None    None  None
 9        IncidentGrade                   VARCHAR  YES  None    None  None
 10       ActionGrouped                   VARCHAR  YES  None    None  None
 11      ActionGranular                   VARCHAR  YES  None    None  None
 12          EntityType  

In [15]:
keys_df = duckdb.query(f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT Id) AS n_distinct_id,
  COUNT(DISTINCT (OrgId, IncidentId)) AS n_distinct_incident,
  COUNT(DISTINCT (OrgId, AlertId)) AS n_distinct_alert
FROM {base}
""").to_df()
keys_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_distinct_id,n_distinct_incident,n_distinct_alert
0,9516837,730778,730778,1265644


In [16]:
duckdb.query(f"""
SELECT
  COUNT(*) AS n_incidents,
  SUM(CASE WHEN n_grades > 1 THEN 1 ELSE 0 END) AS incidents_with_multiple_grades
FROM (
  SELECT OrgId, IncidentId, COUNT(DISTINCT IncidentGrade) AS n_grades
  FROM {base}
  GROUP BY OrgId, IncidentId
)
""").to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_incidents,incidents_with_multiple_grades
0,730778,0.0


In [17]:
evidence_per_alert = duckdb.query(f"""
SELECT
  approx_quantile(cnt, 0.50) AS p50,
  approx_quantile(cnt, 0.90) AS p90,
  approx_quantile(cnt, 0.99) AS p99,
  max(cnt) AS max_evidences_per_alert
FROM (
  SELECT OrgId, AlertId, COUNT(*) AS cnt
  FROM {base}
  GROUP BY OrgId, AlertId
)
""").to_df()
evidence_per_alert

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,p50,p90,p99,max_evidences_per_alert
0,4,12,54,11292


In [18]:
schema_df = duckdb.query(f"DESCRIBE SELECT * FROM {base}").to_df()
cols = schema_df["column_name"].tolist()

# Construye SELECT con % non-null por columna
parts = []
for c in cols:
    parts.append(f"(COUNT({c}) * 100.0 / COUNT(*)) AS '{c}'")

sql = "SELECT " + ",\n  ".join(parts) + f"\nFROM {base}"
completeness_wide = duckdb.query(sql).to_df()

# Pásalo a formato largo (Colonna, Completezza_%)
completeness = completeness_wide.T.reset_index().rename(
    columns={"index": "Colonna", 0: "Completezza_%"}
)
completeness

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Colonna,Completezza_%
0,Id,100.000000
1,OrgId,100.000000
2,IncidentId,100.000000
3,AlertId,100.000000
4,Timestamp,100.000000
5,DetectorId,100.000000
6,AlertTitle,100.000000
7,Category,100.000000
8,MitreTechniques,42.539880
9,IncidentGrade,99.460535


In [19]:
duckdb.query(f"""
SELECT
  COUNT(*) AS n_rows,
  SUM(CASE WHEN EntityType IS NULL THEN 1 ELSE 0 END) AS n_null_entitytype
FROM {base}
""").to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_null_entitytype
0,9516837,0.0


In [20]:
query = f"""
SELECT DISTINCT CountryCode
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE
  CountryCode IS NOT NULL
ORDER BY CountryCode
"""
country_codes_df = duckdb.query(query).to_df()
country_codes_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,CountryCode
0,0
1,1
2,2
3,3
4,4
...,...
231,238
232,239
233,240
234,241


In [21]:
query = f"""
SELECT
  COUNT(DISTINCT OSFamily) AS n_distinct,
  MIN(OSFamily) AS min_cc,
  MAX(OSFamily) AS max_cc
FROM read_csv_auto('{train_path}',
delim='{csv_params.get("sep", ",")}')
WHERE 1=1
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_distinct,min_cc,max_cc
0,6,0,5


In [22]:
query = f"""
SELECT CountryCode, COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
GROUP BY CountryCode
ORDER BY n DESC
LIMIT 20
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,CountryCode,n
0,242,8766912
1,0,173218
2,1,106406
3,2,53671
4,4,40295
5,3,36406
6,5,33464
7,6,22877
8,7,19323
9,8,17934


In [23]:
query = f"""
SELECT
  EXTRACT(year FROM CAST(Timestamp AS TIMESTAMPTZ)) AS year,
  COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE Timestamp IS NOT NULL
GROUP BY year
ORDER BY year
"""
yearly_df = duckdb.query(query).to_df()
yearly_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,year,n
0,2023,179
1,2024,9516658


In [24]:
query = f"""
SELECT
  DATE_TRUNC('month', CAST(Timestamp AS TIMESTAMPTZ)) AS month,
  COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE Timestamp IS NOT NULL
and EXTRACT(year FROM CAST(Timestamp AS TIMESTAMPTZ)) = 2023
GROUP BY month
ORDER BY month
"""
monthly_df = duckdb.query(query).to_df()
monthly_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,n
0,2023-11-01 00:00:00+01:00,11
1,2023-12-01 00:00:00+01:00,168


In [25]:
query = f"""
SELECT
  DATE_TRUNC('month', CAST(Timestamp AS TIMESTAMPTZ)) AS month,
  COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE Timestamp IS NOT NULL
and EXTRACT(year FROM CAST(Timestamp AS TIMESTAMPTZ)) = 2024
GROUP BY month
ORDER BY month
"""
monthly_df = duckdb.query(query).to_df()
monthly_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,n
0,2024-01-01 00:00:00+01:00,310
1,2024-02-01 00:00:00+01:00,297
2,2024-03-01 00:00:00+01:00,235
3,2024-04-01 00:00:00+02:00,287
4,2024-05-01 00:00:00+02:00,848413
5,2024-06-01 00:00:00+02:00,8667116


In [26]:
query = f"""
SELECT
  EXTRACT(year  FROM CAST(Timestamp AS TIMESTAMPTZ)) AS year,
  EXTRACT(month FROM CAST(Timestamp AS TIMESTAMPTZ)) AS month,
  COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE Timestamp IS NOT NULL

GROUP BY year, month
ORDER BY year, month
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,year,month,n
0,2023,11,11
1,2023,12,168
2,2024,1,310
3,2024,2,297
4,2024,3,235
5,2024,4,287
6,2024,5,848413
7,2024,6,8667116


In [27]:
query = f"""
SELECT
  EXTRACT(year  FROM CAST(Timestamp AS TIMESTAMPTZ)) AS year,
  EXTRACT(month FROM CAST(Timestamp AS TIMESTAMPTZ)) AS month,
  EXTRACT(day   FROM CAST(Timestamp AS TIMESTAMPTZ)) AS day

FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE Timestamp IS NOT NULL
and EXTRACT(year FROM CAST(Timestamp AS TIMESTAMPTZ)) IN (2023)
AND EXTRACT(month FROM CAST(Timestamp AS TIMESTAMPTZ)) IN (12)
ORDER   BY month, day
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,year,month,day
0,2023,12,1
1,2023,12,1
2,2023,12,2
3,2023,12,2
4,2023,12,5
...,...,...,...
163,2023,12,29
164,2023,12,30
165,2023,12,30
166,2023,12,30


In [28]:
query = f"""
SELECT
  EXTRACT(year  FROM CAST(Timestamp AS TIMESTAMPTZ)) AS year,
  EXTRACT(month FROM CAST(Timestamp AS TIMESTAMPTZ)) AS month,
  EXTRACT(day   FROM CAST(Timestamp AS TIMESTAMPTZ)) AS day,
  HOUR(CAST(Timestamp AS TIMESTAMPTZ)) AS hour

FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE Timestamp IS NOT NULL
and EXTRACT(year FROM CAST(Timestamp AS TIMESTAMPTZ)) IN (2023)
AND EXTRACT(month FROM CAST(Timestamp AS TIMESTAMPTZ)) IN (12)
AND EXTRACT(day FROM CAST(Timestamp AS TIMESTAMPTZ)) IN (5)
ORDER   BY month, day, hour
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,year,month,day,hour
0,2023,12,5,1
1,2023,12,5,14
2,2023,12,5,15
3,2023,12,5,15
4,2023,12,5,21


In [29]:
query = f"""
SELECT
Timestamp,
EXTRACT(year  FROM CAST(Timestamp AS TIMESTAMPTZ)) AS year,
EXTRACT(month FROM CAST(Timestamp AS TIMESTAMPTZ)) AS month,
EXTRACT(day   FROM CAST(Timestamp AS TIMESTAMPTZ)) AS day,
HOUR(CAST(Timestamp AS TIMESTAMPTZ)) AS hour

FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE Timestamp IS NOT NULL
and EXTRACT(year FROM CAST(Timestamp AS TIMESTAMPTZ)) IN (2023)
AND EXTRACT(month FROM CAST(Timestamp AS TIMESTAMPTZ)) IN (12)
AND EXTRACT(day FROM CAST(Timestamp AS TIMESTAMPTZ)) IN (5)
ORDER   BY month, day, hour
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Timestamp,year,month,day,hour
0,2023-12-05 01:45:58+01:00,2023,12,5,1
1,2023-12-05 14:22:54+01:00,2023,12,5,14
2,2023-12-05 15:29:04+01:00,2023,12,5,15
3,2023-12-05 15:12:14+01:00,2023,12,5,15
4,2023-12-05 21:57:41+01:00,2023,12,5,21


In [30]:
query = f"""
SELECT ActionGrouped, COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
GROUP BY ActionGrouped
ORDER BY n DESC
LIMIT 20
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ActionGrouped,n
0,None,9460773
1,ContainAccount,53760
2,IsolateDevice,2237
3,Stop Virtual Machines,67


In [31]:
query = f"""
SELECT ActionGranular , COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
GROUP BY ActionGranular
ORDER BY n DESC
LIMIT 20
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ActionGranular,n
0,None,9460773
1,update stsrefreshtokenvalidfrom timestamp.,21393
2,account password changed,14059
3,change user password.,13623
4,isolateresponse,2043
5,account disabled,1991
6,disable account.,1143
7,reset user password.,886
8,forcepasswordresetremediation,234
9,quarantinefile,194


In [32]:
query = f"""
SELECT IncidentGrade  , COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
GROUP BY IncidentGrade
ORDER BY n DESC
LIMIT 20
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,IncidentGrade,n
0,BenignPositive,4110817
1,TruePositive,3322713
2,FalsePositive,2031967
3,None,51340


In [5]:
#SuspicionLevel
query = f"""
SELECT SuspicionLevel  , COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
GROUP BY SuspicionLevel
ORDER BY n DESC
LIMIT 20
"""
duckdb.query(query).to_df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,SuspicionLevel,n
0,None,8072708
1,Suspicious,1442614
2,Incriminated,1515


In [15]:
#EntityType
query = f"""
SELECT EntityType  , COUNT(*) AS n
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
GROUP BY EntityType
ORDER BY n DESC
LIMIT 70
"""
duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,EntityType,n
0,Ip,2181194
1,User,1932416
2,MailMessage,1173154
3,Machine,699208
4,File,688402
5,Url,682578
6,CloudLogonRequest,638565
7,Mailbox,483158
8,Process,345732
9,MailCluster,224684


In [14]:
#EntityType
query = f"""
SELECT EntityType  , COUNT(*) AS n
FROM read_csv_auto('{test_path}', delim='{csv_params.get("sep", ",")}')
GROUP BY EntityType
ORDER BY n DESC
LIMIT 70
"""
duckdb.query(query).to_df()

,EntityType,n
0,Ip,929684
1,User,831162
2,MailMessage,479227
3,Machine,348147
4,CloudLogonRequest,298656
5,Url,294913
6,File,290887
7,Mailbox,205624
8,Process,165178
9,MailCluster,96901


In [12]:


# ============================================
# VERIFICAR SI SON IDs (muchos valores únicos)
# ============================================
query_check_ids = f"""
SELECT
    'DetectorId' AS column_name,
    COUNT(DISTINCT DetectorId) AS unique_values,
    COUNT(*) AS total_rows,
    ROUND(COUNT(DISTINCT DetectorId)::FLOAT / COUNT(*) * 100, 2) AS uniqueness_pct,
    MIN(DetectorId) AS min_value,
    MAX(DetectorId) AS max_value
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')

UNION ALL

SELECT
    'OrgId' AS column_name,
    COUNT(DISTINCT OrgId) AS unique_values,
    COUNT(*) AS total_rows,
    ROUND(COUNT(DISTINCT OrgId)::FLOAT / COUNT(*) * 100, 2) AS uniqueness_pct,
    MIN(OrgId) AS min_value,
    MAX(OrgId) AS max_value
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')

UNION ALL

SELECT
    'AlertTitle' AS column_name,
    COUNT(DISTINCT AlertTitle) AS unique_values,
    COUNT(*) AS total_rows,
    ROUND(COUNT(DISTINCT AlertTitle)::FLOAT / COUNT(*) * 100, 2) AS uniqueness_pct,
    MIN(AlertTitle) AS min_value,
    MAX(AlertTitle) AS max_value
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
"""

df_ids = duckdb.query(query_check_ids).to_df()
print("=== VERIFICACIÓN: ¿Son IDs? ===")
print(df_ids)

# INTERPRETACIÓN:
# - Si uniqueness_pct > 90% → Es un ID (no usar log_transform)
# - Si uniqueness_pct < 10% → Es categórica/ordinal (encodear en 3.3)
# - Si 10% < uniqueness_pct < 90% → Verificar distribución

# ============================================
# VERIFICAR DISTRIBUCIÓN (si justifica log)
# ============================================
query_distribution = f"""
SELECT
    'DetectorId' AS column_name,
    MIN(DetectorId) AS min_val,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY DetectorId) AS q25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY DetectorId) AS median,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY DetectorId) AS q75,
    MAX(DetectorId) AS max_val,
    AVG(DetectorId) AS mean_val,
    STDDEV(DetectorId) AS std_val,
    -- Skewness aproximado: si mean >> median → asimétrica positiva
    CASE
        WHEN AVG(DetectorId) > PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY DetectorId) * 1.5
        THEN 'SKEWED_RIGHT (log transform justificado)'
        ELSE 'SYMMETRIC (log transform NO necesario)'
    END AS distribution_shape
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')

UNION ALL

SELECT
    'OrgId' AS column_name,
    MIN(OrgId), PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY OrgId),
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY OrgId),
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY OrgId),
    MAX(OrgId), AVG(OrgId), STDDEV(OrgId),
    CASE WHEN AVG(OrgId) > PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY OrgId) * 1.5
         THEN 'SKEWED_RIGHT' ELSE 'SYMMETRIC' END
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')

UNION ALL

SELECT
    'AlertTitle' AS column_name,
    MIN(AlertTitle), PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY AlertTitle),
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY AlertTitle),
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY AlertTitle),
    MAX(AlertTitle), AVG(AlertTitle), STDDEV(AlertTitle),
    CASE WHEN AVG(AlertTitle) > PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY AlertTitle) * 1.5
         THEN 'SKEWED_RIGHT' ELSE 'SYMMETRIC' END
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
"""

df_dist = duckdb.query(query_distribution).to_df()
print("\n=== DISTRIBUCIÓN: ¿Justifica log transform? ===")
print(df_dist)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== VERIFICACIÓN: ¿Son IDs? ===
  column_name  unique_values  total_rows  uniqueness_pct  min_value  max_value
0  DetectorId           8428     9516837            0.09          0       9522
1       OrgId           5769     9516837            0.06          0       6147
2  AlertTitle          86149     9516837            0.91          0     113174


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== DISTRIBUCIÓN: ¿Justifica log transform? ===
  column_name  min_val   q25  median    q75  max_val     mean_val  \
0  DetectorId        0   2.0     9.0   45.0     9522   110.672426   
1       OrgId        0  10.0    45.0  171.0     6147   181.579950   
2  AlertTitle        0   2.0    11.0  180.0   113174  2947.315112   

        std_val                        distribution_shape  
0    435.103790  SKEWED_RIGHT (log transform justificado)  
1    386.778377                              SKEWED_RIGHT  
2  11461.504048                              SKEWED_RIGHT  


In [13]:


# ============================================
# 1. CONTAR TODOS LOS UNIQUE (no solo top 20)
# ============================================
query_all_unique = f"""
SELECT
    'train' AS dataset,
    COUNT(DISTINCT EntityType) AS unique_values,
    COUNT(*) AS total_rows
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')

UNION ALL

SELECT
    'test' AS dataset,
    COUNT(DISTINCT EntityType) AS unique_values,
    COUNT(*) AS total_rows
FROM read_csv_auto('{test_path}', delim='{csv_params.get("sep", ",")}')
"""

df_unique = duckdb.query(query_all_unique).to_df()
print("=== UNIQUE VALUES POR DATASET ===")
print(df_unique)

# ============================================
# 2. VER TODOS LOS VALORES (SIN LIMIT 20)
# ============================================
query_all_values = f"""
WITH combined AS (
    SELECT EntityType, 'train' AS source
    FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
    UNION ALL
    SELECT EntityType, 'test' AS source
    FROM read_csv_auto('{test_path}', delim='{csv_params.get("sep", ",")}')
)
SELECT
    EntityType,
    COUNT(*) AS total_count,
    SUM(CASE WHEN source = 'train' THEN 1 ELSE 0 END) AS train_count,
    SUM(CASE WHEN source = 'test' THEN 1 ELSE 0 END) AS test_count,
    CASE
        WHEN SUM(CASE WHEN source = 'test' THEN 1 ELSE 0 END) = 0 THEN 'TRAIN_ONLY'
        WHEN SUM(CASE WHEN source = 'train' THEN 1 ELSE 0 END) = 0 THEN 'TEST_ONLY'
        ELSE 'BOTH'
    END AS presence
FROM combined
GROUP BY EntityType
ORDER BY total_count DESC
"""

df_all_values = duckdb.query(query_all_values).to_df()
print(f"\n=== TODOS LOS VALORES ({len(df_all_values)} únicos) ===")
print(df_all_values)

# ============================================
# 3. VERIFICAR CASE SENSITIVITY / WHITESPACE
# ============================================
query_case_whitespace = f"""
WITH combined AS (
    SELECT EntityType FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
    UNION ALL
    SELECT EntityType FROM read_csv_auto('{test_path}', delim='{csv_params.get("sep", ",")}')
)
SELECT
    'Original' AS version,
    COUNT(DISTINCT EntityType) AS unique_values
FROM combined

UNION ALL

SELECT
    'Lowercase' AS version,
    COUNT(DISTINCT LOWER(EntityType)) AS unique_values
FROM combined

UNION ALL

SELECT
    'Trimmed' AS version,
    COUNT(DISTINCT TRIM(EntityType)) AS unique_values
FROM combined

UNION ALL

SELECT
    'Lower + Trimmed' AS version,
    COUNT(DISTINCT TRIM(LOWER(EntityType))) AS unique_values
FROM combined
"""

df_clean = duckdb.query(query_case_whitespace).to_df()
print("\n=== IMPACTO DE LIMPIEZA ===")
print(df_clean)

# ============================================
# 4. DETECTAR VALORES PROBLEMÁTICOS
# ============================================
query_problems = f"""
WITH combined AS (
    SELECT EntityType FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
    UNION ALL
    SELECT EntityType FROM read_csv_auto('{test_path}', delim='{csv_params.get("sep", ",")}')
)
SELECT
    EntityType AS original_value,
    TRIM(LOWER(EntityType)) AS cleaned_value,
    COUNT(*) AS occurrences,
    CASE
        WHEN EntityType != TRIM(EntityType) THEN 'HAS_WHITESPACE'
        WHEN EntityType != LOWER(EntityType) THEN 'HAS_UPPERCASE'
        WHEN EntityType IS NULL THEN 'NULL_VALUE'
        ELSE 'OK'
    END AS issue_type
FROM combined
GROUP BY EntityType
HAVING issue_type != 'OK'
ORDER BY occurrences DESC
"""

df_problems = duckdb.query(query_problems).to_df()
print("\n=== VALORES PROBLEMÁTICOS ===")
print(df_problems if len(
    df_problems) > 0 else "No se encontraron problemas de case/whitespace")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== UNIQUE VALUES POR DATASET ===
  dataset  unique_values  total_rows
0   train             33     9516837
1    test             29     4147992


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== TODOS LOS VALORES (33 únicos) ===
               EntityType  total_count  train_count  test_count    presence
0                      Ip      3110878    2181194.0    929684.0        BOTH
1                    User      2763578    1932416.0    831162.0        BOTH
2             MailMessage      1652381    1173154.0    479227.0        BOTH
3                 Machine      1047355     699208.0    348147.0        BOTH
4                    File       979289     688402.0    290887.0        BOTH
5                     Url       977491     682578.0    294913.0        BOTH
6       CloudLogonRequest       937221     638565.0    298656.0        BOTH
7                 Mailbox       688782     483158.0    205624.0        BOTH
8                 Process       510910     345732.0    165178.0        BOTH
9             MailCluster       321585     224684.0     96901.0        BOTH
10       CloudApplication       311143     216811.0     94332.0        BOTH
11      CloudLogonSession       306844     212382

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== IMPACTO DE LIMPIEZA ===
           version  unique_values
0         Original             33
1        Lowercase             33
2          Trimmed             33
3  Lower + Trimmed             33


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== VALORES PROBLEMÁTICOS ===
           original_value          cleaned_value  occurrences     issue_type
0                      Ip                     ip      3110878  HAS_UPPERCASE
1                    User                   user      2763578  HAS_UPPERCASE
2             MailMessage            mailmessage      1652381  HAS_UPPERCASE
3                 Machine                machine      1047355  HAS_UPPERCASE
4                    File                   file       979289  HAS_UPPERCASE
5                     Url                    url       977491  HAS_UPPERCASE
6       CloudLogonRequest      cloudlogonrequest       937221  HAS_UPPERCASE
7                 Mailbox                mailbox       688782  HAS_UPPERCASE
8                 Process                process       510910  HAS_UPPERCASE
9             MailCluster            mailcluster       321585  HAS_UPPERCASE
10       CloudApplication       cloudapplication       311143  HAS_UPPERCASE
11      CloudLogonSession      cloudlogonsess

In [16]:


# Query para calcular skewness de TODAS las columnas numéricas
query = f"""
WITH numeric_stats AS (
    SELECT
        'DetectorId' AS column_name,
        AVG(DetectorId) AS mean_val,
        STDDEV(DetectorId) AS std_val,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY DetectorId) AS median_val,
        MIN(DetectorId) AS min_val,
        MAX(DetectorId) AS max_val,
        COUNT(DISTINCT DetectorId) AS unique_count,
        COUNT(*) AS total_count
    FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')

    UNION ALL

    SELECT
        'OrgId' AS column_name,
        AVG(OrgId), STDDEV(OrgId),
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY OrgId),
        MIN(OrgId), MAX(OrgId),
        COUNT(DISTINCT OrgId), COUNT(*)
    FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')

    UNION ALL

    SELECT
        'AlertTitle' AS column_name,
        AVG(AlertTitle), STDDEV(AlertTitle),
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY AlertTitle),
        MIN(AlertTitle), MAX(AlertTitle),
        COUNT(DISTINCT AlertTitle), COUNT(*)
    FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')

    -- Agregar más columnas int64 aquí...
)
SELECT
    column_name,
    mean_val,
    median_val,
    std_val,
    min_val,
    max_val,
    unique_count,
    total_count,
    ROUND(unique_count::FLOAT / total_count * 100, 2) AS uniqueness_pct,
    ROUND(max_val / NULLIF(min_val, 0), 2) AS range_ratio,
    -- Skewness aproximado (si mean >> median → skewed right)
    CASE
        WHEN mean_val > median_val * 1.5 THEN 'SKEWED_RIGHT'
        WHEN mean_val < median_val * 0.5 THEN 'SKEWED_LEFT'
        ELSE 'SYMMETRIC'
    END AS distribution_shape,
    -- ¿Es candidato para log transform?
    CASE
        WHEN uniqueness_pct > 50 THEN 'NO: ID (high uniqueness)'
        WHEN unique_count < 100 THEN 'NO: CATEGORICAL (low unique count)'
        WHEN distribution_shape = 'SYMMETRIC' THEN 'NO: symmetric distribution'
        WHEN range_ratio < 10 THEN 'NO: narrow range'
        ELSE 'YES: log transform candidate'
    END AS log_transform_decision
FROM numeric_stats
ORDER BY
    CASE
        WHEN log_transform_decision LIKE 'YES%' THEN 1
        ELSE 2
    END,
    uniqueness_pct DESC
"""

df_candidates = duckdb.query(query).to_df()
print("=== CANDIDATOS PARA LOG TRANSFORM ===")
print(df_candidates)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== CANDIDATOS PARA LOG TRANSFORM ===
  column_name     mean_val  median_val       std_val  min_val  max_val  \
0  AlertTitle  2947.315112        11.0  11461.504048        0   113174   
1  DetectorId   110.672426         9.0    435.103790        0     9522   
2       OrgId   181.579950        45.0    386.778377        0     6147   

   unique_count  total_count  uniqueness_pct  range_ratio distribution_shape  \
0         86149      9516837            0.91          NaN       SKEWED_RIGHT   
1          8428      9516837            0.09          NaN       SKEWED_RIGHT   
2          5769      9516837            0.06          NaN       SKEWED_RIGHT   

         log_transform_decision  
0  YES: log transform candidate  
1  YES: log transform candidate  
2  YES: log transform candidate  
